# Extract Boundary Tokens - Simple Version
Based on camelbert_inference_for_export.ipynb

In [ ]:
from google.colab import drive
import os, time
drive.mount('/content/drive')
time.sleep(2)
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print(f"Working directory: {os.getcwd()}")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/khabar-segmentation


In [ ]:
!pip install transformers torch tqdm -q
print("Dependencies OK")

Dependencies OK


In [ ]:
import json
import torch
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Load model from Drive
model_path = Path('checkpoints/camelbert_binary_classification_final')
print(f"Loading model from {model_path}...")
tokenizer = AutoTokenizer.from_pretrained(str(model_path))
model = AutoModelForTokenClassification.from_pretrained(str(model_path))
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
print("Model loaded")

Loading model from checkpoints/camelbert_binary_classification_final...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded


In [ ]:
# Load corpus - UPDATED FOR alDarrab
corpus_file = Path('data/processed/alDarrab_clean.txt')
print(f"Loading corpus from {corpus_file}...")
with open(corpus_file, encoding='utf-8') as f:
    text = f.read()
print(f"Corpus: {len(text):,} chars")

In [ ]:
# Process FULL TEXT in chunks with ENHANCED format
from tqdm import tqdm
from scipy.special import softmax  # For computing probabilities

print("Running inference on FULL CORPUS...")
print(f"Text length: {len(text):,} chars")

all_tokens = []
all_predictions = []
all_offsets = []
all_probabilities = []  # NEW: capture confidences

CHUNK_SIZE = 500  # Slightly less than max to avoid edge effects
chunks_processed = 0

# Split text into overlapping chunks
for start_char in tqdm(range(0, len(text), CHUNK_SIZE), desc="Processing chunks"):
    end_char = min(start_char + CHUNK_SIZE + 50, len(text))  # Small overlap
    chunk_text = text[start_char:end_char]

    # Encode chunk
    encoded = tokenizer(
        chunk_text,
        return_tensors='pt',
        return_offsets_mapping=True,
        truncation=False,  # NO truncation - process full chunk
        padding=False,
    )

    # Run inference on chunk
    with torch.no_grad():
        if torch.cuda.is_available():
            outputs = model(
                input_ids=encoded['input_ids'].cuda(),
                attention_mask=encoded['attention_mask'].cuda()
            )
        else:
            outputs = model(**encoded)
        logits = outputs.logits[0]  # Shape: (seq_len, num_classes)

    # Get predictions and probabilities (softmax of logits)
    preds = np.argmax(logits.cpu().numpy(), axis=-1)
    probs = softmax(logits.cpu().numpy(), axis=-1)  # Probabilities for all classes
    # Extract probability of boundary class (pred=1)
    boundary_probs = probs[:, 1].tolist()  # Probability that token is boundary
    
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
    offsets = encoded['offset_mapping'][0].numpy()

    # Avoid duplicate tokens from overlap
    if chunks_processed > 0 and len(all_tokens) > 0:
        # Skip first 5 tokens of new chunk to avoid duplicates
        preds = preds[5:]
        tokens = tokens[5:]
        offsets = offsets[5:]
        boundary_probs = boundary_probs[5:]

    all_tokens.extend(tokens)
    all_predictions.extend(preds.tolist())
    all_offsets.extend(offsets.tolist())
    all_probabilities.extend(boundary_probs)  # NEW

    chunks_processed += 1

print(f"\nProcessing complete")
print(f"  Chunks processed: {chunks_processed}")
print(f"  Total tokens: {len(all_tokens):,}")
print(f"  Boundary tokens: {sum(all_predictions):,}")
print(f"  Percentage: {100 * sum(all_predictions) / len(all_predictions):.2f}%")

In [ ]:
# Extract boundary tokens
print("Extracting boundary tokens...")
boundary_tokens = []
boundary_indices = []

for idx, (token, pred) in enumerate(zip(all_tokens, all_predictions)):
    if pred == 1:  # Boundary token
        boundary_tokens.append(token)
        boundary_indices.append(idx)

print(f"Extracted: {len(boundary_tokens):,} boundary tokens")

Extracting boundary tokens...
Extracted: 17,011 boundary tokens


In [14]:
print("\nFirst 50 boundary tokens:")
for i, token in enumerate(boundary_tokens[:100], 1):
    print(f"{i:3d}. {token}")


First 50 boundary tokens:
  1. .
  2. أخبرنا
  3. أبو
  4. بكر
  5. محمد
  6. بن
  7. عبدالله
  8. الأرد
  9. ##يش
 10. ##ائي
 11. في
 12. المسجد
 13. الحرام
 14. قراءة
 15. عليه
 16. قال
 17. :
 18. حدثنا
 19. أبو
 20. الحسن
 21. محمد
 22. بن
 23. حبيب
 24. بني
 25. ##ساب
 26. ##ور
 27. قال
 28. :
 29. حدثنا
 30. أبو
 31. إسحاق
 32. إبراهيم
 33. بن
 34. محمد
 35. بن
 36. يزيد
 37. النس
 38. ##في
 39. بمر
 40. ##و
 41. قال
 42. :
 43. حدثنا
 44. أبو
 45. عبيد
 46. الله
 47. خت
 48. ##ن
 49. أبي
 50. بكر
 51. الورا
 52. ##ق
 53. قال
 54. :
 55. سئل
 56. أبو
 57. بكر
 58. .
 59. أخبرنا
 60. محمد
 61. بن
 62. أحمد
 63. قال
 64. :
 65. حدثنا
 66. الحسن
 67. بن
 68. محمد
 69. بن
 70. حبيب
 71. قال
 72. حدثنا
 73. أبو
 74. عبدالله
 75. محمد
 76. بن
 77. عبد
 78. الله
 79. بن
 80. أحمد
 81. الخطيب
 82. السيد
 83. ##ابي
 84. بز
 85. ##وزن
 86. قال
 87. :
 88. حدثنا
 89. أبو
 90. قريش
 91. محمد
 92. بن
 93. جمعة
 94. بن
 95. خلف
 96. الحافظ
 97. قال
 98. :
 99. حدثنا
100. محمد


In [ ]:
# Save ENHANCED RAW INFERENCE format (compatible with convert_boundary_tokens_direct.py)
print("\nSaving raw inference to JSON...")

# Verify all lists have same length
assert len(all_tokens) == len(all_predictions) == len(all_offsets) == len(all_probabilities), \
    f"Length mismatch: tokens={len(all_tokens)}, preds={len(all_predictions)}, offsets={len(all_offsets)}, probs={len(all_probabilities)}"

results = {
    'metadata': {
        'corpus': 'alDarrab_clean.txt',
        'corpus_size_chars': len(text),
        'total_tokens': len(all_tokens),
        'model': 'camelbert_binary_classification_final',
        'processing_method': 'Chunked inference with overlap',
        'chunk_size': CHUNK_SIZE,
        'chunks_processed': chunks_processed,
        'boundary_tokens_count': sum(all_predictions),
        'boundary_percentage': round(100 * sum(all_predictions) / len(all_tokens), 2),
    },
    'inference_results': {
        'total_tokens': len(all_tokens),
        'tokens': all_tokens,
        'offsets': all_offsets,  # [[start, end], [start, end], ...]
        'predictions': all_predictions,  # [0, 1, 0, 1, ...]
        'probabilities': all_probabilities,  # Probability of boundary (class 1)
    }
}

output_file = Path('results/camelbert_alDarrab_raw_inference.json')
output_file.parent.mkdir(parents=True, exist_ok=True)
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

file_size = output_file.stat().st_size / (1024 * 1024)
print(f"Saved: {output_file}")
print(f"Size: {file_size:.2f} MB")
print(f"\nInference Summary:")
print(f"  Total tokens: {len(all_tokens):,}")
print(f"  Boundary tokens: {sum(all_predictions):,}")
print(f"  Boundary percentage: {100 * sum(all_predictions) / len(all_tokens):.2f}%")
print(f"  Mean boundary probability: {np.mean(all_probabilities):.4f}")
print(f"\nNext Step:")
print(f"  Download {output_file} to local machine")
print(f"  Run: python scripts/convert_boundary_tokens_direct.py")
print(f"  Output: results/camelbert_alDarrab_char_boundaries.json")
print(f"Done!")

In [ ]:
# Validate saved format
print("=== VALIDATION: Raw Inference Format ===\n")

# Reload and verify
with open(output_file, 'r', encoding='utf-8') as f:
    saved_data = json.load(f)

inf = saved_data['inference_results']
print(f"Format Check:")
print(f"  'tokens' key: {len(inf['tokens'])} items ✓")
print(f"  'offsets' key: {len(inf['offsets'])} items ✓")
print(f"  'predictions' key: {len(inf['predictions'])} items ✓")
print(f"  'probabilities' key: {len(inf['probabilities'])} items ✓")

print(f"\nSample Data (first 5 tokens):")
for i in range(min(5, len(inf['tokens']))):
    tok = inf['tokens'][i]
    off = inf['offsets'][i]
    pred = inf['predictions'][i]
    prob = inf['probabilities'][i]
    print(f"  [{i}] token={tok!r:15s} offset={off} pred={pred} prob={prob:.4f}")

print(f"\nBoundary Token Samples (pred=1):")
boundary_indices = [i for i, p in enumerate(inf['predictions']) if p == 1]
for i in boundary_indices[:5]:
    tok = inf['tokens'][i]
    off = inf['offsets'][i]
    prob = inf['probabilities'][i]
    print(f"  [{i}] token={tok!r:15s} offset={off} prob={prob:.4f}")

print(f"\n✓ Format validated and ready for clustering post-processing")